# Sign type dataset: feature statistics and class separability

This notebook analyses the reviewed/merged `sign_type_classifier` dataset without running SAM3. It covers class balance, geometry, position, mask shape, colour/context features, SAM3 scores, PCA, centroid distances, and initial SAM3 sorting errors.

Run `reindex_dataset.py` first so `manifest.jsonl` contains the manually corrected labels, including `not_a_sign`.

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from PIL import Image

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 11, 'axes.labelsize': 10})
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)

## Configuration
Set `DATASET_DIR` to the directory produced by the final reindex/merge. `PREDICTIONS_JSONL` is optional and can point to output from `sign_type_classifier/scripts/predict.py`.

In [ ]:
DATASET_DIR = Path('/home/a60116606/git_repo/noise_seg/pipeline_v0/output/sign_type_dataset_merged')
MANIFEST_PATH = DATASET_DIR / 'manifest.jsonl'
DATASET_MANIFEST_PATH = DATASET_DIR / 'dataset_manifest.json'
PREDICTIONS_JSONL: Path | None = None
REPORT_DIR = DATASET_DIR / 'feature_analysis'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_ORDER = [
    'height_restriction_sign_at_underground',
    'height_restriction_barrel',
    'underground_parking_sign',
    'overhead_traffic_sign',
    'side_plate',
    'induction_sign',
    'not_a_sign',
]
DISPLAY_NAMES = {
    'height_restriction_sign_at_underground': 'height sign',
    'height_restriction_barrel': 'height bar',
    'underground_parking_sign': 'underground sign',
    'overhead_traffic_sign': 'overhead sign',
    'side_plate': 'side plate',
    'induction_sign': 'induction sign',
    'not_a_sign': 'not a sign',
}
CLASS_COLORS = dict(zip(CLASS_ORDER, plt.get_cmap('tab10').colors[:len(CLASS_ORDER)]))

assert MANIFEST_PATH.is_file(), f'Manifest not found: {MANIFEST_PATH}'
print('DATASET_DIR:', DATASET_DIR)
print('REPORT_DIR:', REPORT_DIR)

## Load and validate the feature table

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

records = load_jsonl(MANIFEST_PATH)
dataset_meta = json.loads(DATASET_MANIFEST_PATH.read_text(encoding='utf-8')) if DATASET_MANIFEST_PATH.is_file() else {}
feature_names = list(dataset_meta.get('numeric_feature_names', []))
if not feature_names and records:
    feature_names = list(records[0].get('numeric_features', {}).keys())
if not feature_names:
    raise ValueError('numeric_feature_names are absent from dataset_manifest.json')

rows = []
for record in records:
    feature_map = record.get('numeric_features')
    if isinstance(feature_map, dict) and all(name in feature_map for name in feature_names):
        values = [feature_map[name] for name in feature_names]
    else:
        values = record.get('numeric_feature_vector', [])
    if len(values) != len(feature_names):
        raise ValueError(f"{record.get('sample_id')}: {len(values)} features, expected {len(feature_names)}")
    row = {
        'sample_id': str(record['sample_id']),
        'source_id': str(record['source_id']),
        'label': str(record['label']),
        'initial_label': str(record.get('initial_label', record['label'])),
        'split': str(record.get('split', 'unassigned')),
        'context_rgb': str(record.get('context_rgb', '')),
        'preview': str(record.get('preview', '')),
        'sam3_score': float(record.get('sam3_score', np.nan)),
        'sam3_margin': float(record.get('sam3_margin', np.nan)),
    }
    row.update(dict(zip(feature_names, map(float, values))))
    rows.append(row)

df = pd.DataFrame(rows)
unknown_labels = sorted(set(df['label']) - set(CLASS_ORDER))
if unknown_labels:
    raise ValueError(f'Unknown labels: {unknown_labels}')
df['label'] = pd.Categorical(df['label'], categories=CLASS_ORDER, ordered=True)
feature_df = df[feature_names].apply(pd.to_numeric, errors='coerce')
finite_ratio = np.isfinite(feature_df.to_numpy(dtype=np.float64)).mean()
print(f'samples={len(df)}, source frames={df.source_id.nunique()}, features={len(feature_names)}')
print(f'finite feature values={finite_ratio:.3%}')
display(df[['sample_id', 'source_id', 'label', 'initial_label', 'split']].head())

In [ ]:
FEATURE_GROUPS = {
    'geometry_position': [name for name in feature_names if name.startswith(('bbox_', 'mask_', 'touches_', 'distance_'))],
    'sam3_scores': [name for name in feature_names if name.startswith('sam_')],
    'object_appearance': [name for name in feature_names if name.startswith('object_')],
    'context_appearance': [name for name in feature_names if name.startswith('context_')],
}
quality = pd.DataFrame({
    'group': [group for group, names in FEATURE_GROUPS.items() for _ in names],
    'feature': [name for names in FEATURE_GROUPS.values() for name in names],
}).groupby('group').size().rename('features').to_frame()
quality.loc['all', 'features'] = len(feature_names)
display(quality.astype(int))

bad_columns = [name for name in feature_names if not np.isfinite(feature_df[name]).all()]
constant_columns = [name for name in feature_names if feature_df[name].nunique(dropna=True) <= 1]
print('non-finite columns:', bad_columns)
print('constant columns:', constant_columns)

## Class and split balance
A class should ideally have examples from several independent source frames in every split. A large number of crops from one frame is not equivalent to a large dataset.

In [ ]:
class_stats = pd.DataFrame({
    'samples': df.groupby('label', observed=False).size(),
    'source_frames': df.groupby('label', observed=False)['source_id'].nunique(),
    'manually_changed': df.assign(changed=df.initial_label != df.label.astype(str)).groupby('label', observed=False)['changed'].sum(),
})
class_stats['samples_per_frame'] = class_stats['samples'] / class_stats['source_frames'].clip(lower=1)
display(class_stats)
class_stats.to_csv(REPORT_DIR / 'class_statistics.csv')

split_counts = pd.crosstab(df['label'], df['split']).reindex(CLASS_ORDER, fill_value=0)
display(split_counts)
ax = split_counts.plot(kind='bar', stacked=True, figsize=(12, 5), color=['#4c78a8', '#f2cf5b', '#e45756'])
ax.set_title('Samples per class and split')
ax.set_xlabel('')
ax.set_ylabel('samples')
ax.set_xticklabels([DISPLAY_NAMES[name] for name in CLASS_ORDER], rotation=35, ha='right')
plt.tight_layout()
plt.show()

## Geometry and absolute image position
The image coordinate origin is at the top-left. The Y axis is inverted in the position plot to match the camera image. Point size represents normalized bbox area.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for label in CLASS_ORDER:
    part = df[df.label == label]
    if part.empty:
        continue
    sizes = 15 + 1200 * np.clip(part['bbox_area'].to_numpy(), 0, 0.08)
    axes[0].scatter(part['bbox_center_x'], part['bbox_center_y'], s=sizes, alpha=0.45, color=CLASS_COLORS[label], label=DISPLAY_NAMES[label])
    axes[1].scatter(part['bbox_width'], part['bbox_height'], s=25, alpha=0.45, color=CLASS_COLORS[label])
axes[0].set(xlim=(0, 1), ylim=(1, 0), xlabel='bbox center X', ylabel='bbox center Y', title='Position in full camera frame')
axes[1].set(xlabel='normalized bbox width', ylabel='normalized bbox height', title='Detection size and aspect')
axes[0].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
SELECTED_GEOMETRY = [
    'bbox_bottom', 'bbox_width', 'bbox_height', 'bbox_area',
    'bbox_aspect_ratio', 'mask_bbox_fill_ratio',
]
selected = [name for name in SELECTED_GEOMETRY if name in feature_names]
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, feature in zip(axes.flat, selected):
    values = [df.loc[df.label == label, feature].dropna().to_numpy() for label in CLASS_ORDER]
    boxes = ax.boxplot(values, patch_artist=True, showfliers=False)
    for patch, label in zip(boxes['boxes'], CLASS_ORDER):
        patch.set_facecolor(CLASS_COLORS[label]); patch.set_alpha(0.65)
    ax.set_title(feature)
    ax.set_xticks(range(1, len(CLASS_ORDER) + 1), [DISPLAY_NAMES[name] for name in CLASS_ORDER], rotation=40, ha='right', fontsize=8)
for ax in axes.flat[len(selected):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Colour, brightness, and edge structure
Means are useful for a quick view; histograms still remain available to the model and are included in the feature ranking below.

In [ ]:
APPEARANCE_FEATURES = [
    'object_rgb_mean_r', 'object_rgb_mean_g', 'object_rgb_mean_b',
    'object_gray_mean', 'object_gray_std', 'object_edge_density',
    'context_gray_mean', 'context_gray_std', 'context_edge_density',
]
appearance = [name for name in APPEARANCE_FEATURES if name in feature_names]
appearance_means = df.groupby('label', observed=False)[appearance].mean().reindex(CLASS_ORDER)
fig, ax = plt.subplots(figsize=(12, 5))
image = ax.imshow(appearance_means.to_numpy(), aspect='auto', cmap='viridis')
ax.set_yticks(range(len(CLASS_ORDER)), [DISPLAY_NAMES[name] for name in CLASS_ORDER])
ax.set_xticks(range(len(appearance)), appearance, rotation=40, ha='right')
ax.set_title('Mean object/context appearance features')
fig.colorbar(image, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## Rank individual features by class separation
The Fisher score is between-class variance divided by within-class variance. A high score means that class means are far apart relative to their internal spread. It is a univariate diagnostic, not a substitute for validation metrics.

In [ ]:
def fisher_score(values: np.ndarray, labels: np.ndarray) -> float:
    finite = np.isfinite(values)
    values = values[finite]
    labels = labels[finite]
    if values.size < 2 or np.nanstd(values) < 1e-12:
        return 0.0
    global_mean = values.mean()
    between = 0.0
    within = 0.0
    for label in CLASS_ORDER:
        group = values[labels == label]
        if group.size == 0:
            continue
        between += group.size * (group.mean() - global_mean) ** 2
        within += np.square(group - group.mean()).sum()
    return float(between / (within + 1e-12))

labels_np = df['label'].astype(str).to_numpy()
feature_scores = pd.Series({
    name: fisher_score(df[name].to_numpy(dtype=np.float64), labels_np)
    for name in feature_names
}, name='fisher_score').sort_values(ascending=False)
display(feature_scores.head(30).to_frame())
feature_scores.to_csv(REPORT_DIR / 'feature_fisher_scores.csv')

top = feature_scores.head(25).sort_values()
fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(top.index, top.values, color='#4c78a8')
ax.set_title('Top individual features by Fisher separation score')
ax.set_xlabel('between-class / within-class variance')
plt.tight_layout()
plt.show()

In [ ]:
TOP_FEATURE_COUNT = min(18, len(feature_scores))
top_features = feature_scores.head(TOP_FEATURE_COUNT).index.tolist()
means = df.groupby('label', observed=False)[top_features].mean().reindex(CLASS_ORDER)
std = df[top_features].std().replace(0, 1)
global_mean = df[top_features].mean()
standardized_means = (means - global_mean) / std
fig, ax = plt.subplots(figsize=(14, 6))
image = ax.imshow(standardized_means.to_numpy(), aspect='auto', cmap='coolwarm', vmin=-2.5, vmax=2.5)
ax.set_yticks(range(len(CLASS_ORDER)), [DISPLAY_NAMES[name] for name in CLASS_ORDER])
ax.set_xticks(range(len(top_features)), top_features, rotation=45, ha='right')
ax.set_title('Class means for top features (global standard deviations)')
fig.colorbar(image, ax=ax, shrink=0.8, label='z-score')
plt.tight_layout()
plt.show()

## PCA and pairwise class distances
PCA gives a two-dimensional view of all non-constant numeric features. Overlap in PCA does not prove the classes are inseparable because PCA is unsupervised and the EfficientNet image branch is not represented here.

In [ ]:
usable_features = [name for name in feature_names if name not in constant_columns]
X = df[usable_features].to_numpy(dtype=np.float64)
column_medians = np.nanmedian(X, axis=0)
bad = ~np.isfinite(X)
X[bad] = np.take(column_medians, np.where(bad)[1])
mean = X.mean(axis=0)
scale = X.std(axis=0)
scale[scale < 1e-8] = 1.0
Z = (X - mean) / scale
_, singular_values, vt = np.linalg.svd(Z, full_matrices=False)
pca = Z @ vt[:2].T
explained = np.square(singular_values) / np.square(singular_values).sum()

fig, ax = plt.subplots(figsize=(10, 7))
for label in CLASS_ORDER:
    mask = labels_np == label
    ax.scatter(pca[mask, 0], pca[mask, 1], s=25, alpha=0.55, color=CLASS_COLORS[label], label=DISPLAY_NAMES[label])
ax.set_xlabel(f'PC1 ({explained[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({explained[1]:.1%} variance)')
ax.set_title('PCA of all numeric features')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
centroids = np.stack([Z[labels_np == label].mean(axis=0) if np.any(labels_np == label) else np.full(Z.shape[1], np.nan) for label in CLASS_ORDER])
distances = np.sqrt(np.square(centroids[:, None, :] - centroids[None, :, :]).mean(axis=2))
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(distances, cmap='magma')
ax.set_xticks(range(len(CLASS_ORDER)), [DISPLAY_NAMES[name] for name in CLASS_ORDER], rotation=40, ha='right')
ax.set_yticks(range(len(CLASS_ORDER)), [DISPLAY_NAMES[name] for name in CLASS_ORDER])
for row in range(len(CLASS_ORDER)):
    for column in range(len(CLASS_ORDER)):
        if np.isfinite(distances[row, column]):
            ax.text(column, row, f'{distances[row, column]:.2f}', ha='center', va='center', color='white' if distances[row, column] > np.nanmedian(distances) else 'black', fontsize=8)
ax.set_title('RMS distance between standardized class centroids')
fig.colorbar(image, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## Numeric-only nearest-centroid baseline
This intentionally simple baseline standardizes features and computes class centroids using only `train`. It then evaluates `test` (or `val` when test is absent). It is not the final MLP, but its grouped holdout confusion matrix is a useful concrete check of numeric separability.

In [ ]:
train_mask = df['split'].astype(str).to_numpy() == 'train'
evaluation_split = 'test' if np.any(df['split'].astype(str).to_numpy() == 'test') else 'val'
evaluation_mask = df['split'].astype(str).to_numpy() == evaluation_split
if not np.any(train_mask) or not np.any(evaluation_mask):
    print('Train/evaluation split is missing; run reindex_dataset.py before this diagnostic.')
else:
    raw_numeric = df[usable_features].to_numpy(dtype=np.float64)
    train_X = raw_numeric[train_mask].copy()
    eval_X = raw_numeric[evaluation_mask].copy()
    train_medians = np.nanmedian(np.where(np.isfinite(train_X), train_X, np.nan), axis=0)
    train_medians[~np.isfinite(train_medians)] = 0.0
    train_bad = ~np.isfinite(train_X)
    eval_bad = ~np.isfinite(eval_X)
    train_X[train_bad] = np.take(train_medians, np.where(train_bad)[1])
    eval_X[eval_bad] = np.take(train_medians, np.where(eval_bad)[1])
    train_labels = labels_np[train_mask]
    eval_labels = labels_np[evaluation_mask]
    train_mean = train_X.mean(axis=0)
    train_scale = train_X.std(axis=0)
    train_scale[train_scale < 1e-8] = 1.0
    train_Z = (train_X - train_mean) / train_scale
    eval_Z = (eval_X - train_mean) / train_scale
    missing_train_classes = [label for label in CLASS_ORDER if not np.any(train_labels == label)]
    if missing_train_classes:
        print('Missing train classes:', missing_train_classes)
    available_classes = [label for label in CLASS_ORDER if np.any(train_labels == label)]
    train_centroids = np.stack([train_Z[train_labels == label].mean(axis=0) for label in available_classes])
    squared_distances = np.square(eval_Z[:, None, :] - train_centroids[None, :, :]).mean(axis=2)
    predictions = np.asarray(available_classes)[squared_distances.argmin(axis=1)]
    confusion = pd.crosstab(
        pd.Categorical(eval_labels, categories=CLASS_ORDER),
        pd.Categorical(predictions, categories=CLASS_ORDER),
        dropna=False,
    ).reindex(index=CLASS_ORDER, columns=CLASS_ORDER, fill_value=0)
    normalized = confusion.div(confusion.sum(axis=1).replace(0, 1), axis=0)
    accuracy = float(np.mean(predictions == eval_labels))
    print(f'nearest-centroid {evaluation_split} accuracy: {accuracy:.3%} ({len(eval_labels)} samples)')
    display(confusion)
    fig, ax = plt.subplots(figsize=(8, 7))
    image = ax.imshow(normalized.to_numpy(), vmin=0, vmax=1, cmap='Blues')
    ax.set_xticks(range(len(CLASS_ORDER)), [DISPLAY_NAMES[name] for name in CLASS_ORDER], rotation=40, ha='right')
    ax.set_yticks(range(len(CLASS_ORDER)), [DISPLAY_NAMES[name] for name in CLASS_ORDER])
    ax.set_xlabel('nearest centroid prediction')
    ax.set_ylabel('manual class')
    ax.set_title(f'Numeric-only baseline on {evaluation_split}')
    fig.colorbar(image, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()

## Initial SAM3 label versus manual label
This table shows where class-specific prompts confuse one sign type with another. `not_a_sign` in a row indicates the corresponding SAM3 prompt frequently produces false positives.

In [ ]:
initial_order = [name for name in CLASS_ORDER if name != 'not_a_sign']
initial_vs_manual = pd.crosstab(df['initial_label'], df['label']).reindex(index=initial_order, columns=CLASS_ORDER, fill_value=0)
row_totals = initial_vs_manual.sum(axis=1).replace(0, 1)
initial_normalized = initial_vs_manual.div(row_totals, axis=0)
display(initial_vs_manual)
initial_vs_manual.to_csv(REPORT_DIR / 'sam3_initial_vs_manual_counts.csv')

fig, ax = plt.subplots(figsize=(10, 6))
image = ax.imshow(initial_normalized.to_numpy(), vmin=0, vmax=1, cmap='Blues')
ax.set_xticks(range(len(CLASS_ORDER)), [DISPLAY_NAMES[name] for name in CLASS_ORDER], rotation=40, ha='right')
ax.set_yticks(range(len(initial_order)), [DISPLAY_NAMES[name] for name in initial_order])
ax.set_xlabel('manual class')
ax.set_ylabel('initial SAM3 class')
ax.set_title('Initial SAM3 sorting normalized by prompt class')
fig.colorbar(image, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## Inspect examples by class or feature extreme
Use this to verify that a statistical difference is real rather than an annotation or cropping artifact.

In [ ]:
def show_examples(
    *,
    label: str | None = None,
    sort_feature: str = 'bbox_area',
    largest: bool = True,
    count: int = 8,
) -> None:
    if sort_feature not in df.columns:
        raise KeyError(sort_feature)
    selected = df if label is None else df[df.label == label]
    selected = selected.sort_values(sort_feature, ascending=not largest).head(count)
    if selected.empty:
        print('No matching samples')
        return
    columns = min(4, len(selected))
    rows = int(np.ceil(len(selected) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(4.5 * columns, 3.8 * rows), squeeze=False)
    for ax, (_, item) in zip(axes.flat, selected.iterrows()):
        relative = item['preview'] or item['context_rgb']
        path = DATASET_DIR / relative
        if path.is_file():
            ax.imshow(Image.open(path).convert('RGB'))
        ax.set_title(f"{DISPLAY_NAMES[str(item['label'])]}\n{sort_feature}={item[sort_feature]:.4g}", fontsize=9)
        ax.axis('off')
    for ax in axes.flat[len(selected):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_examples(label=None, sort_feature='bbox_area', largest=True, count=8)

## Optional: compare trained image, numeric, and ensemble predictions
Set `PREDICTIONS_JSONL` in the configuration cell to the JSONL created by `predict.py`.

In [ ]:
def confusion_from_probability_maps(predictions: list[dict], field: str) -> np.ndarray:
    class_to_id = {name: index for index, name in enumerate(CLASS_ORDER)}
    confusion = np.zeros((len(CLASS_ORDER), len(CLASS_ORDER)), dtype=np.int64)
    for item in predictions:
        target = item.get('target')
        probabilities = item.get(field, {})
        if target not in class_to_id or not probabilities:
            continue
        predicted = max(CLASS_ORDER, key=lambda name: float(probabilities.get(name, 0.0)))
        confusion[class_to_id[target], class_to_id[predicted]] += 1
    return confusion

if PREDICTIONS_JSONL is None:
    print('PREDICTIONS_JSONL is not configured; skipping trained-model diagnostics.')
else:
    predictions = load_jsonl(Path(PREDICTIONS_JSONL))
    fields = ['image_probabilities', 'numeric_probabilities', 'probabilities']
    titles = ['EfficientNet context branch', 'Numeric feature branch', 'Ensemble']
    fig, axes = plt.subplots(1, 3, figsize=(19, 5))
    for ax, field, title in zip(axes, fields, titles):
        confusion = confusion_from_probability_maps(predictions, field)
        normalized = confusion / np.maximum(confusion.sum(axis=1, keepdims=True), 1)
        image = ax.imshow(normalized, vmin=0, vmax=1, cmap='Blues')
        ax.set_title(title)
        ax.set_xticks(range(len(CLASS_ORDER)), [DISPLAY_NAMES[name] for name in CLASS_ORDER], rotation=45, ha='right', fontsize=7)
        ax.set_yticks(range(len(CLASS_ORDER)), [DISPLAY_NAMES[name] for name in CLASS_ORDER], fontsize=7)
        ax.set_xlabel('prediction')
        ax.set_ylabel('target')
    fig.colorbar(image, ax=axes, shrink=0.75)
    plt.show()

## Quick interpretation checklist

- Prefer **source frame count** over raw crop count when deciding whether a class has enough data.
- Strong separation in `bbox_center_y`, `bbox_bottom`, or `distance_to_top` supports the numeric position branch.
- Strong separation in object/context colour or edge features supports the appearance part of the numeric branch.
- A high Fisher score with obviously wrong gallery examples usually means annotation leakage or a preprocessing artifact.
- Close class centroids and overlapping PCA clusters suggest that context imagery is needed; inspect EfficientNet metrics after training.
- A high `not_a_sign` fraction for one initial prompt identifies a prompt or geometry filter that should be tightened.
- Final decisions must be based on grouped validation/test metrics, not PCA alone.